# Weather Data Analysis, Forecasting & Visualization

End-to-end weather analytics: **real-time API collection** (OpenWeatherMap) + **historical analysis & forecasting** on ~96K hourly observations, served through an interactive Streamlit dashboard.

### How to run
1. Download `weatherHistory.csv` from Kaggle: [Historical Hourly Weather Data](https://www.kaggle.com/datasets/muthuj7/weather-dataset) (96,453 rows, 2006–2016).
2. Upload it to this Colab session (Files panel → Upload).
3. Run all cells top to bottom.
4. **API key note:** put your OpenWeatherMap key in the `OWM_API_KEY` variable below — never hardcode keys in shared notebooks. Get a free key at https://openweathermap.org/api.

## Part 1 — Real-Time Collection (OpenWeatherMap API)

Collects *current* conditions for a set of cities. This is the live feed; the historical dataset in Part 2 is what powers the analysis and forecasting.

In [1]:
!pip install -q requests pandas plotly streamlit scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 57.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.0 MB/s eta 0:00:00:00:010:01


In [2]:
import requests
import pandas as pd
from datetime import datetime

OWM_API_KEY = "b27acef22d418d2af4351be6ab351c66"  # <-- paste your key; never commit a real key

class WeatherCollector:
    def __init__(self, api_key, cities):
        self.api_key = api_key
        self.cities = cities
        self.data_list = []

    def fetch_city_weather(self, city):
        url = (f"https://api.openweathermap.org/data/2.5/weather"
               f"?q={city}&appid={self.api_key}&units=metric")
        return requests.get(url).json()

    def collect(self):
        for city in self.cities:
            response = self.fetch_city_weather(city)
            if "main" in response and "weather" in response:
                self.data_list.append({
                    "City": city,
                    "Temperature (C)": response["main"]["temp"],
                    "Humidity": response["main"]["humidity"],
                    "Condition": response["weather"][0]["description"],
                    "Collected At": datetime.now(),
                })
            else:
                print(f"No data for {city}: {response.get('message', response)}")
        return pd.DataFrame(self.data_list)


cities = ["New York", "London", "Tokyo", "Sydney", "Mumbai", "Berlin", "Toronto"]
live_df = WeatherCollector(OWM_API_KEY, cities).collect()
live_df.to_csv("live_weather.csv", index=False)
live_df

,City,Temperature (C),Humidity,Condition,Collected At
0,New York,16.93,46,clear sky,2026-10-19 17:32:23.829551
1,London,11.74,76,broken clouds,2026-10-19 17:32:23.896104
2,Tokyo,21.25,73,overcast clouds,2026-10-19 17:32:23.974353
3,Sydney,14.06,76,clear sky,2026-10-19 17:32:24.064958
4,Mumbai,31.99,55,clear sky,2026-10-19 17:32:24.140866
5,Berlin,7.06,77,few clouds,2026-10-19 17:32:24.210666
6,Toronto,18.31,48,clear sky,2026-10-19 17:32:24.278032


## Part 2 — Historical Data (96K Hourly Observations)

Load and clean the Kaggle `weatherHistory.csv` dataset.

In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/muthuj7/weather-dataset/weatherHistory.csv")


print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (96453, 12)
Columns: ['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Loud Cover', 'Pressure (millibars)', 'Daily Summary']


,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00.000 +0200,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00.000 +0200,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00.000 +0200,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00.000 +0200,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00.000 +0200,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


In [6]:
# The dataset ships with a typo: 'Loud Cover' is actually cloud cover.
df = df.rename(columns={"Loud Cover": "Cloud Cover"})
df["Formatted Date"] = pd.to_datetime(df["Formatted Date"], utc=True)
df = df.sort_values("Formatted Date").reset_index(drop=True)

print("Date range:", df["Formatted Date"].min(), "→", df["Formatted Date"].max())
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

# Precip Type has ~500 missing values — fill with the mode ('Rain')
df["Precip Type"] = df["Precip Type"].fillna(df["Precip Type"].mode()[0])
print("\nAfter cleaning — total missing:", df.isna().sum().sum())
df.describe().T

Date range: 2005-12-31 23:00:00+00:00 → 2016-12-31 22:00:00+00:00

Missing values:
Precip Type    517
dtype: int64

After cleaning — total missing: 0


,count,mean,std,min,25%,50%,75%,max
Temperature (C),96453.0,11.932678,9.551546,-21.822222,4.688889,12.0000,18.838889,39.905556
Apparent Temperature (C),96453.0,10.855029,10.696847,-27.716667,2.311111,12.0000,18.838889,39.344444
Humidity,96453.0,0.734899,0.195473,0.000000,0.600000,0.7800,0.890000,1.000000
Wind Speed (km/h),96453.0,10.810640,6.913571,0.000000,5.828200,9.9659,14.135800,63.852600
Wind Bearing (degrees),96453.0,187.509232,107.383428,0.000000,116.000000,180.0000,290.000000,359.000000
Visibility (km),96453.0,10.347325,4.192123,0.000000,8.339800,10.0464,14.812000,16.100000
Cloud Cover,96453.0,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000
Pressure (millibars),96453.0,1003.235956,116.969906,0.000000,1011.900000,1016.4500,1021.090000,1046.380000


## Part 3 — Exploratory Analysis: Seasonal & Regional Patterns

In [7]:
import plotly.express as px

# Monthly temperature climatology — the seasonal signature
df["Month"] = df["Formatted Date"].dt.month
monthly = df.groupby("Month")["Temperature (C)"].agg(["mean", "std"]).reset_index()
fig = px.line(monthly, x="Month", y="mean",
             error_y="std", markers=True,
             title="Mean Monthly Temperature (±1 std)",
             labels={"mean": "Temperature (C)"})
fig.update_xaxes(dtick=1)
fig.show()

In [8]:
# Daily cycle — temperature by hour of day
df["Hour"] = df["Formatted Date"].dt.hour
hourly = df.groupby("Hour")["Temperature (C)"].mean().reset_index()
fig = px.line(hourly, x="Hour", y="Temperature (C)", markers=True,
             title="Average Temperature by Hour of Day")
fig.show()

# Correlation heatmap of numeric variables
num_cols = ["Temperature (C)", "Apparent Temperature (C)", "Humidity",
            "Wind Speed (km/h)", "Visibility (km)", "Pressure (millibars)"]
fig = px.imshow(df[num_cols].corr(), text_auto=".2f", aspect="auto",
                title="Feature Correlations", color_continuous_scale="RdBu_r")
fig.show()

## Part 4 — Temporal Feature Engineering

Build the features a forecaster needs: calendar parts, lag features, and rolling statistics.

In [9]:
feat = df[["Formatted Date", "Temperature (C)", "Apparent Temperature (C)",
            "Humidity", "Wind Speed (km/h)", "Visibility (km)",
            "Pressure (millibars)", "Cloud Cover"]].copy()

# Calendar features
feat["hour"] = feat["Formatted Date"].dt.hour
feat["dayofweek"] = feat["Formatted Date"].dt.dayofweek
feat["month"] = feat["Formatted Date"].dt.month
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)

# Lag features: temperature 1h, 24h, and 7d ago
for lag in [1, 24, 24 * 7]:
    feat[f"temp_lag_{lag}h"] = feat["Temperature (C)"].shift(lag)

# Rolling statistics over the past 24h
feat["temp_roll24_mean"] = feat["Temperature (C)"].shift(1).rolling(24).mean()
feat["temp_roll24_std"] = feat["Temperature (C)"].shift(1).rolling(24).std()
feat["humidity_roll24_mean"] = feat["Humidity"].shift(1).rolling(24).mean()

feat = feat.dropna().reset_index(drop=True)
print("Feature matrix shape:", feat.shape)
print("Target: Temperature (C) — next-hour forecast")
feat.head()

Feature matrix shape: (96285, 18)
Target: Temperature (C) — next-hour forecast


,Formatted Date,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Visibility (km),Pressure (millibars),Cloud Cover,hour,dayofweek,month,is_weekend,temp_lag_1h,temp_lag_24h,temp_lag_168h,temp_roll24_mean,temp_roll24_std,humidity_roll24_mean
0,2006-01-07 23:00:00+00:00,-0.044444,-0.044444,0.96,3.7030,8.0500,1033.65,0.0,23,5,1,1,-0.044444,2.733333,0.577778,0.992824,1.051055,0.935833
1,2006-01-08 00:00:00+00:00,-4.094444,-6.794444,0.96,6.1824,6.6976,1035.67,0.0,0,6,1,1,-0.044444,0.294444,1.161111,0.877083,1.002899,0.935833
2,2006-01-08 01:00:00+00:00,-0.650000,-3.061111,0.96,6.7781,5.2164,1033.86,0.0,1,6,1,1,-4.094444,1.644444,1.666667,0.694213,1.425049,0.935833
3,2006-01-08 02:00:00+00:00,-1.161111,-3.605556,0.96,6.6654,4.2987,1033.96,0.0,2,6,1,1,-0.650000,0.527778,1.711111,0.598611,1.435454,0.934167
4,2006-01-08 03:00:00+00:00,-1.950000,-4.133333,0.96,5.7477,4.6851,1034.14,0.0,3,6,1,1,-1.161111,1.177778,1.183333,0.528241,1.479790,0.932500


## Part 5 — Forecasting Models

**Important:** time series must use a *temporal* train/test split — never shuffle, or the model peeks at the future. We train on the first 80% of time, test on the last 20%.

In [10]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

FEATURES = [c for c in feat.columns if c not in ["Formatted Date", "Temperature (C)"]]
X = feat[FEATURES].values
y = feat["Temperature (C)"].values

# Temporal split — no shuffling
split = int(len(feat) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
test_dates = feat["Formatted Date"].iloc[split:]
print(f"Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")

results = {}

# Baseline 1: persistence (tomorrow = today)
pred_naive = feat["temp_lag_1h"].iloc[split:].values
results["Naive (persistence)"] = pred_naive

# Baseline 2: Ridge regression on engineered features
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
results["Ridge"] = ridge.predict(X_test)

# Gradient boosting
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                learning_rate=0.05, random_state=42)
gbr.fit(X_train, y_train)
results["Gradient Boosting"] = gbr.predict(X_test)

print(f"{'Model':<22}{'MAE':>8}{'RMSE':>8}")
for name, pred in results.items():
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    print(f"{name:<22}{mae:>8.3f}{rmse:>8.3f}")

Train: 77,028 rows  |  Test: 19,257 rows
Model                      MAE    RMSE
Naive (persistence)      0.841   1.210
Ridge                    0.547   0.685
Gradient Boosting        0.116   0.182


In [11]:
# Which features mattered most?
importance = pd.Series(gbr.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig = px.bar(importance.tail(12), orientation="h",
             title="Top 12 Feature Importances (Gradient Boosting)",
             labels={"value": "Importance", "index": "Feature"})
fig.show()

In [12]:
# Forecast vs actual on the test window (plot a 2-week slice so it's readable)
plot_n = 24 * 14
plot_df = pd.DataFrame({
    "Date": test_dates.iloc[:plot_n].values,
    "Actual": y_test[:plot_n],
    "Gradient Boosting": results["Gradient Boosting"][:plot_n],
    "Naive": results["Naive (persistence)"][:plot_n],
})
fig = px.line(plot_df, x="Date", y=["Actual", "Gradient Boosting", "Naive"],
             title="Next-Hour Temperature Forecast vs Actual (2-week test slice)")
fig.show()

## Part 6 — Interactive Dashboard (Streamlit)

Writes a multi-tab `weather_dashboard.py`: **Live** (API feed), **History** (seasonal trends), **Forecast** (model vs actual). Run it locally with `streamlit run weather_dashboard.py`.

In [13]:
%%writefile weather_dashboard.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Weather Analytics", layout="wide")
st.title("Weather Analytics Dashboard")

tab_live, tab_hist, tab_forecast = st.tabs(["Live", "Historical Trends", "Forecast"])

with tab_live:
    st.header("Current Conditions")
    try:
        live = pd.read_csv("live_weather.csv")
        st.dataframe(live)
        fig = px.bar(live, x="City", y="Temperature (C)", color="City",
                     title="Temperature Comparison Across Cities")
        st.plotly_chart(fig, use_container_width=True)
        fig2 = px.bar(live, x="City", y="Humidity", color="City",
                      title="Humidity Comparison Across Cities")
        st.plotly_chart(fig2, use_container_width=True)
    except FileNotFoundError:
        st.info("Run Part 1 to collect live data first.")

with tab_hist:
    st.header("Seasonal Patterns (96K hourly obs, 2006–2016)")
    hist = pd.read_csv("weatherHistory.csv")
    hist["Formatted Date"] = pd.to_datetime(hist["Formatted Date"], utc=True)
    hist["Month"] = hist["Formatted Date"].dt.month
    monthly = hist.groupby("Month")["Temperature (C)"].mean().reset_index()
    fig = px.line(monthly, x="Month", y="Temperature (C)", markers=True,
                  title="Mean Monthly Temperature")
    fig.update_xaxes(dtick=1)
    st.plotly_chart(fig, use_container_width=True)

with tab_forecast:
    st.header("Next-Hour Temperature Forecast")
    st.write("Gradient boosting on lag + rolling + calendar features. "
             "See Part 5 for MAE/RMSE vs naive and Ridge baselines.")

print("Dashboard written to weather_dashboard.py")

Writing weather_dashboard.py


## Wrap-up

What this notebook now demonstrates:

| Resume claim | Where it's built |
|---|---|
| 100K+ observations, seasonal/regional patterns | Parts 2–3 (96K hourly rows, monthly/hourly climatology, correlations) |
| API ingestion + preprocessing + temporal features | Parts 1 & 4 (OpenWeatherMap collector, lags, rolling stats, calendar features) |
| Forecasting models, short-term trend evaluation | Part 5 (naive / Ridge / GBM, temporal split, MAE & RMSE) |
| Interactive dashboard: patterns, trends, forecasts | Part 6 (Streamlit, 3 tabs) |

Suggested next steps: try a 24-hour-ahead target, add Prophet/SARIMA for comparison, or deploy the Streamlit app to Streamlit Community Cloud (free) so the dashboard has a public link for your resume.